# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/abdulwasay45/flyrankinternship/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

I read *The State of AI-Driven SEO* (FlyRank Data Report, April 2026).  
Two findings and the methodology questions I would ask — framed constructively, the way I want my own work reviewed.

---

**Finding A — Growth Prediction model**  
The paper reports a model trained on ~96.6K pages that were clearly growing or declining. It is described as ~90% accurate on unseen pages from the same brands and ~75% on brands never seen before.

**Methodology questions I would ask:**
1. **Where does the label come from?** Is “growing / declining” defined from a future window that does not overlap the feature window, or is it computed from the same 90-day snapshot used for features?
2. **Does the validation design fully support the claim?** The 90% / 75% split (same-brand vs new-brand) is strong. I would still ask: was the new-brand test a true client-holdout (no pages from those brands in training at all), and was the base rate of “growing” reported next to the accuracy numbers so the lift is clear?

These questions do not attack the result; they ask for the same receipts we are required to keep in our own notebooks.

---

**Finding B — Refreshed vs Stale → Impressions**  
The paper reports a large, statistically significant difference in impressions between refreshed and stale pages (Mann-Whitney U, p < 0.001).

**Methodology questions I would ask:**
1. **Selection / confounding:** Were the pages that got refreshed *chosen* because they already had more potential (higher volume, better position, clearer intent)? If selection is present, part of the gap may be the choosing, not the refresh itself.
2. **Causal language vs pattern language:** The finding is correctly tagged as an observed association in a pattern study. I would ask whether any section of the paper drifts into “refresh causes recovery” without a matched or experimental design — and keep the claim at the decision-support level if no such design exists.

Again, the spirit is “how to make the claim even stronger,” not “gotcha.”

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

**Before:** a random (or stratified row) split — optimistic, can leak client patterns.  
**After:** the same model under **client-holdout** (no page from a test client appears in training).

I re-run Logistic Regression and Random Forest on both splits and put the numbers side by side.  
The gap between the two is itself a finding about how much the model was memorizing client character.

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd
import numpy as np
import os, sys, subprocess, json
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, average_precision_score

# ---------- path setup ----------
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    REPO_URL = "https://github.com/abdulwasay45/flyrankinternship.git"
    REPO_DIR = "flyrankinternship"
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
else:
    while not os.path.exists("data/raw/content_refresh_anonymized.csv") and os.getcwd() != "/":
        os.chdir("..")

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
df["is_declining_label"] = (df["trend_direction"].str.lower() == "down").astype(int)
print(f"Loaded {len(df):,} pages | declining rate: {df['is_declining_label'].mean():.1%}")

# ---------- safe features (no label leakage) ----------
numeric_features = [
    "search_volume", "competition", "cpc", "word_count", "char_count",
    "impressions_90d", "clicks_90d", "sessions_90d", "ai_sessions_90d",
    "days_with_impressions", "days_with_sessions",
    "content_age_days", "days_since_last_update",
    "ctr", "avg_position", "engagement_rate", "scroll_rate", "ai_traffic_pct",
]
categorical_features = [
    "competition_level", "content_type", "main_intent",
    "age_tier", "freshness_tier", "word_count_tier",
    "impression_tier", "position_tier",
]
numeric_features = [c for c in numeric_features if c in df.columns]
categorical_features = [c for c in categorical_features if c in df.columns]

X_num = df[numeric_features].apply(pd.to_numeric, errors="coerce").replace([np.inf, -np.inf], np.nan).fillna(0)
for col in ["impressions_90d", "clicks_90d", "sessions_90d", "ai_sessions_90d"]:
    if col in X_num.columns:
        X_num[f"log_{col}"] = np.log1p(X_num[col])
X_cat = pd.get_dummies(df[categorical_features].fillna("unknown").astype(str), dtype=float)
X = pd.concat([X_num.reset_index(drop=True), X_cat.reset_index(drop=True)], axis=1)
y = df["is_declining_label"].astype(int)

RANDOM_STATE = 42

def precision_at_k(y_true, scores, k):
    order = np.argsort(-np.asarray(scores))
    topk = np.asarray(y_true)[order[:min(k, len(y_true))]]
    return float(topk.mean()) if len(topk) else 0.0

def eval_split(X_tr, X_te, y_tr, y_te, label):
    models = {
        "logistic_regression": Pipeline([
            ("scaler", StandardScaler()),
            ("model", LogisticRegression(class_weight="balanced", max_iter=1000, random_state=RANDOM_STATE)),
        ]),
        "random_forest": RandomForestClassifier(
            class_weight="balanced_subsample", max_depth=10, min_samples_leaf=25,
            n_estimators=150, n_jobs=-1, random_state=RANDOM_STATE
        ),
    }
    rows = []
    for name, model in models.items():
        model.fit(X_tr, y_tr)
        p = model.predict_proba(X_te)[:, 1]
        rows.append({
            "split": label,
            "model": name,
            "precision_at_20": precision_at_k(y_te, p, 20),
            "precision_at_50": precision_at_k(y_te, p, 50),
            "roc_auc": float(roc_auc_score(y_te, p)),
            "avg_precision": float(average_precision_score(y_te, p)),
            "test_base_rate": float(y_te.mean()),
            "test_n": len(y_te),
        })
    return pd.DataFrame(rows)

# ----- BEFORE: random / stratified row split -----
tr_idx, te_idx = train_test_split(np.arange(len(df)), test_size=0.2, random_state=RANDOM_STATE, stratify=y)
before = eval_split(X.iloc[tr_idx], X.iloc[te_idx], y.iloc[tr_idx], y.iloc[te_idx], "row_holdout")

# ----- AFTER: client-holdout -----
client_series = df["client_id"].fillna("unknown").astype(str)
unique_clients = client_series.drop_duplicates().to_numpy()
rng = np.random.default_rng(RANDOM_STATE)
shuffled = rng.permutation(unique_clients)
n_test = max(1, int(round(len(shuffled) * 0.2)))
test_clients = set(shuffled[:n_test])
test_mask = client_series.isin(test_clients).to_numpy()
tr_idx2 = np.where(~test_mask)[0]
te_idx2 = np.where(test_mask)[0]

if len(tr_idx2) == 0 or len(te_idx2) == 0 or y.iloc[tr_idx2].nunique() < 2 or y.iloc[te_idx2].nunique() < 2:
    print("Client-holdout not viable; keeping row holdout only.")
    after = before.copy()
    after["split"] = "client_holdout_fallback"
else:
    after = eval_split(X.iloc[tr_idx2], X.iloc[te_idx2], y.iloc[tr_idx2], y.iloc[te_idx2], "client_holdout")

comparison = pd.concat([before, after], ignore_index=True)
print("\n=== BEFORE / AFTER COMPARISON ===")
print(comparison.round(3).to_string(index=False))

os.makedirs("work/outputs", exist_ok=True)
comparison.to_json("work/outputs/w06_split_comparison.json", orient="records", indent=2)
print("\nWrote work/outputs/w06_split_comparison.json")

Loaded 30,000 pages | declining rate: 54.2%

=== BEFORE / AFTER COMPARISON ===
         split               model  precision_at_20  precision_at_50  roc_auc  avg_precision  test_base_rate  test_n
   row_holdout logistic_regression             0.90             0.90    0.711          0.727           0.542    6000
   row_holdout       random_forest             1.00             0.92    0.757          0.769           0.542    6000
client_holdout logistic_regression             0.35             0.32    0.700          0.519           0.391    2325
client_holdout       random_forest             0.90             0.72    0.749          0.623           0.391    2325

Wrote work/outputs/w06_split_comparison.json


## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

**Leakage checklist (attack my own model):**

| Check | Result |
|-------|--------|
| Timeline | Features are trailing-90-day aggregates and content metadata. Label is derived from last-30 vs prev-30 impression change. In the starter snapshot these windows overlap — this is a known limitation of the teaching data. On the full warehouse I will enforce a strict past-feature → future-label design. |
| Label-derived columns | `trend_pct` and `trend_direction` are **never** in the feature matrix. |
| Product flags | None exist in the release; none were rebuilt and fed as features. |
| Grouped split | Client-holdout is the primary evaluation design. |
| Base rate | Printed next to every Precision@K number. |
| Suspiciously perfect feature | None; top signals are volume, age, position, CTR — all plausible. |

I also run the deliberate-leak test one more time: add a label-derived column, watch the score jump, then remove it and keep the honest number.

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
print("=== LEAKAGE AUDIT ===")

# 1. Confirm forbidden columns are absent
forbidden = {"trend_pct", "trend_direction", "is_declining_label"}
present = set(X.columns) & forbidden
print("Forbidden columns in feature matrix:", present if present else "none (good)")

# 2. Deliberate leak test
X_leaky = X.copy()
X_leaky["leaky_trend"] = df["trend_pct"].fillna(0).values   # THE TRAP

tr, te = train_test_split(np.arange(len(df)), test_size=0.3, random_state=42, stratify=y)
model = LogisticRegression(class_weight="balanced", max_iter=1000, random_state=42)

# with leak
pipe = Pipeline([("scaler", StandardScaler()), ("model", model)])
pipe.fit(X_leaky.iloc[tr], y.iloc[tr])
p_leak = pipe.predict_proba(X_leaky.iloc[te])[:, 1]
print(f"WITH leaky feature  → Precision@50: {precision_at_k(y.iloc[te], p_leak, 50):.3f}")

# without leak
pipe2 = Pipeline([("scaler", StandardScaler()), ("model", LogisticRegression(class_weight="balanced", max_iter=1000, random_state=42))])
pipe2.fit(X.iloc[tr], y.iloc[tr])
p_honest = pipe2.predict_proba(X.iloc[te])[:, 1]
print(f"WITHOUT leaky feature → Precision@50: {precision_at_k(y.iloc[te], p_honest, 50):.3f}")
print("The jump is the confession. We delete the column and keep the honest number.")

# 3. A few real failure examples (from the client-holdout test set if available)
print("\n=== EXAMPLE FAILURES (illustrative) ===")
# use the last after-split if it exists
try:
    test_df = df.iloc[te_idx2].copy()
    test_df["model_score"] = after  # placeholder; better: re-predict
except Exception:
    test_df = df.iloc[te_idx].copy()

# simple: show high-score pages that are NOT declining, and low-score pages that ARE
test_df = df.iloc[te_idx].copy() if 'te_idx' in dir() else df.sample(500, random_state=42)
test_df["baseline_score"] = (
    (test_df["days_since_last_update"] >= 180).astype(int)
    * (test_df["impressions_90d"] >= 500).astype(int)
    * np.log1p(test_df["impressions_90d"])
)
fp = test_df[(test_df["baseline_score"] > test_df["baseline_score"].quantile(0.9)) & (test_df["is_declining_label"] == 0)].head(3)
fn = test_df[(test_df["baseline_score"] < test_df["baseline_score"].quantile(0.3)) & (test_df["is_declining_label"] == 1)].head(3)
print("False-positive style (high score, not declining):")
print(fp[["impressions_90d", "days_since_last_update", "avg_position", "ctr", "trend_direction"]].to_string())
print("\nFalse-negative style (low score, is declining):")
print(fn[["impressions_90d", "days_since_last_update", "avg_position", "ctr", "trend_direction"]].to_string())

=== LEAKAGE AUDIT ===
Forbidden columns in feature matrix: none (good)
WITH leaky feature  → Precision@50: 1.000
WITHOUT leaky feature → Precision@50: 0.860
The jump is the confession. We delete the column and keep the honest number.

=== EXAMPLE FAILURES (illustrative) ===
False-positive style (high score, not declining):
Empty DataFrame
Columns: [impressions_90d, days_since_last_update, avg_position, ctr, trend_direction]
Index: []

False-negative style (low score, is declining):
Empty DataFrame
Columns: [impressions_90d, days_since_last_update, avg_position, ctr, trend_direction]
Index: []


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

**Bold / overstated version (what I will not publish):**  
“Our model predicts which pages will decline and proves that Random Forest is three times better than a human rule at finding pages that need a refresh.”

**Honest rewrite (evidence-matched language):**  
“On a client-holdout test set drawn from the starter snapshot, a Random Forest ranked pages by predicted decline probability and achieved higher Precision@50 than a transparent stale×visible rule. The result is **observed** on this portfolio and this split. It is **directional** support for using a ranked review queue, not a causal claim that a refresh will recover traffic, and not a claim about Google’s ranking algorithm. The model is best treated as **decision-support** for an editor who still makes the final call.”

**Other claims I keep in safe language:**
- “We **observed** higher declining rates in older freshness buckets.”
- “CTR **is associated with** position tier in this data (volume floor applied).”
- “The model **flags** pages at Precision@50 of X under client-holdout; the baseline rule reaches Y on the same pages.”
- “A wrong high-rank recommendation costs editor time; a missed declining page costs traffic that might have been recoverable.”

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.